# Fib (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Fib_python.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `Fib`: l'algoritmo in Python, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Fib_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Fib_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno i nomi dello pseudocodice. Due eccezioni in Python, dove `from` e `next` hanno un significato speciale: `Hanoi` usa `frm` al posto di `from` e `IterativeFib` usa `nxt` al posto di `next`.

## Strumenti: contatori, stack e traccia

La cella definisce i contatori (`cnt`, `calls`, `max_depth`), lo stack delle chiamate (`push` e `pop`) e le funzioni che stampano la traccia (`event`, `say`) e la tabella dei passi (`print_steps`). Basta eseguirla una volta.

In [1]:
cnt = [0] * 20      # cnt[r] = numero di esecuzioni della riga r (righe numerate come nelle dispense)
calls = 0           # numero di chiamate = nodi dell'albero delle chiamate
max_depth = 0       # profondita' della ricorsione = altezza massima dello stack
mults = 0           # moltiplicazioni (FastPower)
moves = 0           # mosse (Hanoi)
ev = 0              # contatore degli eventi della traccia
stack = []          # lo stack delle chiamate: gli argomenti delle chiamate attive
seen = [0] * 64     # seen[x] = numero di chiamate con argomento x
trace = False       # True: stampa la traccia dell'esecuzione


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # inizia una chiamata: il suo frame va in cima allo stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # la chiamata termina: il suo frame esce dallo stack
    stack.pop()


def event(text):                   # riga numerata della traccia, con lo stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Riga | Esecuzioni")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Passi totali S = %d" % sum(cnt))

## Numeri di Fibonacci: Fib

**Input:** un numero naturale $n$. **Output:** $F(n)$, con $F(0) = 0$, $F(1) = 1$ e $F(n) = F(n-1) + F(n-2)$. Algoritmo ricorsivo naive: $C(n) = 2F(n+1) - 1$ chiamate e $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ passi.

Pseudocodice: sezione 14.5 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
def fib(n):
    push(n); cnt[2] += 1
    if n <= 1:
        cnt[3] += 1
        if trace: say("Fib(%d)" % n, "caso base, restituisce %d" % n)
        pop()
        return n
    if trace: say("Fib(%d)" % n, "chiama Fib(%d)" % (n - 1))
    x = fib(n - 1)                                 # le due chiamate sono separate solo per stampare la traccia
    if trace: say("Fib(%d)" % n, "chiama Fib(%d)" % (n - 2))
    y = fib(n - 2)
    cnt[5] += 1
    if trace: say("Fib(%d)" % n, "restituisce %d + %d = %d" % (x, y, x + y))
    pop()
    return x + y

In [3]:
def fib_value(k):                                  # F(k) con un ciclo, per controllare
    a, b = 0, 1
    for _ in range(k):
        a, b = b, a + b
    return a


# Esempio delle dispense: l'esecuzione di Fib(4), in ordine di visita in profondita'
n = 4
reset()
trace = True
print("%-28s| %s" % ("chiamata", "azione"))
r = fib(n)
trace = False
print("F(%d) = %d, chiamate C(%d) = %d = 2F(%d) - 1, profondita' = %d" % (n, r, n, calls, n + 1, max_depth))

# Fib(5): gli stessi sottoproblemi sono risolti piu' volte
print()
n = 5
reset()
r = fib(n)
print("Fib(%d) = %d: %d chiamate, profondita' %d, S = %d passi" % (n, r, calls, max_depth, sum(cnt)))
print("  k | chiamate di Fib(k)")
for k in range(n, -1, -1):
    print("%3d | %d" % (k, seen[k]))
print_steps(2, 3, 5)

# Numero di chiamate e di passi: C(n) = 2F(n+1) - 1 e S(n) = 4F(n+1) - 2
print()
print("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | profondita'")
for n in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30]:
    reset()
    r = fib(n)
    print("%3d | %6d | %8d | %11d | %8d | %11d | %d" % (n, r, calls, 2 * fib_value(n + 1) - 1, sum(cnt), 4 * fib_value(n + 1) - 2, max_depth))

chiamata                    | azione
Fib(4)                      | chiama Fib(3)
  Fib(3)                    | chiama Fib(2)
    Fib(2)                  | chiama Fib(1)
      Fib(1)                | caso base, restituisce 1
    Fib(2)                  | chiama Fib(0)
      Fib(0)                | caso base, restituisce 0
    Fib(2)                  | restituisce 1 + 0 = 1
  Fib(3)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(3)                    | restituisce 1 + 1 = 2
Fib(4)                      | chiama Fib(2)
  Fib(2)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(2)                    | chiama Fib(0)
    Fib(0)                  | caso base, restituisce 0
  Fib(2)                    | restituisce 1 + 0 = 1
Fib(4)                      | restituisce 2 + 1 = 3
F(4) = 3, chiamate C(4) = 9 = 2F(5) - 1, profondita' = 4

Fib(5) = 5: 15 chiamate, profondita' 5, S = 30 passi
  k | chiamate di

**Prova tu.** Cambia $n$ e guarda quanto crescono le chiamate.

In [4]:
n = 6        # <-- cambia n (con n > 35 ci vuole un po')
reset()
r = fib(n)
print("F(%d) = %d, chiamate = %d, S = %d" % (n, r, calls, sum(cnt)))

F(6) = 8, chiamate = 25, S = 50


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).